[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/07_tensorflow_practice/07_tensorflow_practice_solutions.ipynb)

# 07. TensorFlow / Keras 실습 — 연습 문제 해설

[07_tensorflow_practice.ipynb](07_tensorflow_practice.ipynb) 끝의 연습 문제 3개에 대한 정답 코드와
설명입니다. 먼저 본편에서 직접 풀어본 뒤 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q tensorflow

import tensorflow as tf
print("TensorFlow version:", tf.__version__)


## 문제 1. `y = 3x - 1`, learning_rate 0.1로 수렴 속도 비교

원래 예제(`y = 2x + 3`, lr=0.01)는 200 epoch에도 완전히 수렴하지 않는 반면, learning_rate를 올리면
훨씬 빠르게 수렴합니다. 다만 lr을 너무 올리면 발산할 수 있다는 점도 같이 확인해보면 좋습니다.

In [ ]:
x = tf.constant([1.0, 2.0, 3.0, 4.0])
y = tf.constant([2.0, 5.0, 8.0, 11.0])   # y = 3x - 1

W = tf.Variable(0.0)
b = tf.Variable(0.0)
optimizer = tf.optimizers.SGD(learning_rate=0.1)

for epoch in range(200):
    with tf.GradientTape() as tape:
        y_pred = W * x + b
        cost = tf.reduce_mean((y_pred - y) ** 2)
    dW, db = tape.gradient(cost, [W, b])
    optimizer.apply_gradients(zip([dW, db], [W, b]))
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  cost={cost.numpy():.4f}  W={W.numpy():.3f}  b={b.numpy():.3f}")

print(f"최종: W={W.numpy():.3f}, b={b.numpy():.3f} (목표: W=3, b=-1)")
# lr=0.01일 때보다 훨씬 적은 epoch에서 목표값(3, -1)에 근접합니다.


## 문제 2. XOR 은닉층 뉴런 수 8 → 2

은닉 뉴런이 2개만 있어도 XOR을 표현할 수 있는 이론적 최소치이긴 하지만, 랜덤 초기화에 따라
학습이 잘 안 되고 local minimum에 걸리는 경우가 자주 발생합니다. 여러 번 실행해서 accuracy가
들쭉날쭉한 것을 확인해보세요 — `04_neural_networks`에서 다룬 "은닉층이 있어야 XOR을 풀 수 있지만,
뉴런 수가 너무 적으면 학습이 불안정하다"는 내용과 연결됩니다.

In [ ]:
x_xor = tf.constant([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = tf.constant([[0.0], [1.0], [1.0], [0.0]])

results = []
for trial in range(5):
    tf.keras.utils.set_random_seed(trial)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(2, activation="relu", input_shape=(2,)),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(x_xor, y_xor, epochs=500, verbose=0)
    _, acc = model.evaluate(x_xor, y_xor, verbose=0)
    results.append(acc)

print("5회 시도 accuracy:", [round(a, 2) for a in results])
# 뉴런 8개일 때보다 성공률이 낮고, 시도마다 결과가 달라지는 것을 확인할 수 있습니다.


## 문제 3. `tf.Variable` vs `tf.constant` — gradient 추적 대상

`tf.GradientTape()`는 기본적으로 `tf.Variable`만 자동으로 추적합니다. `tf.constant`에 대한
gradient를 구하려면 `tape.watch()`로 명시적으로 추적 대상에 등록해야 합니다 — 등록하지 않으면
`None`이 반환됩니다.

In [ ]:
w_var = tf.Variable(2.0)
x_const = tf.constant(3.0)

with tf.GradientTape() as tape:
    y = w_var * x_const

grads = tape.gradient(y, [w_var, x_const])
print("w_var에 대한 gradient:", grads[0].numpy())   # dy/dw = x = 3.0
print("x_const에 대한 gradient:", grads[1])          # None — 기본적으로 추적하지 않음

# tape.watch()로 명시적으로 등록하면 constant도 추적 가능
with tf.GradientTape() as tape:
    tape.watch(x_const)
    y = w_var * x_const

grads = tape.gradient(y, [w_var, x_const])
print("watch() 이후 x_const에 대한 gradient:", grads[1].numpy())   # dy/dx = w = 2.0
